# 20 — Prompt Optimization and Evaluation


**Learning reference:** [BrightPool — Prompt Engineering Course](https://github.com/BrightPool/udemy-prompt-engineering-course)

**Topics:** Prompt optimization, A/B testing, rule-based
evaluation, weighted scoring and held-out testing.

## Objective

Evaluate whether structured instructions improve
Gemini's compliance with social media writing requirements.

Two prompt designs are compared:

- **Baseline:** General social media writing instructions.
- **Optimized:** Explicit rules for factual accuracy,
  length, hashtags, punctuation and tone.

The experiment uses eight fictional business updates,
divided into development and test datasets.

### Workflow

1. Define the task and evaluation criteria.
2. Compare baseline and optimized prompts.
3. Evaluate outputs on development cases.
4. Evaluate both prompts on held-out cases.
5. Inspect factual and editorial limitations.

The objective is to measure observable improvements
rather than assume detailed prompts are always better.



## 1. Experimental Design

**Prompt optimization** improves instructions to achieve
a measurable objective.

**A/B testing** compares two prompt versions against
identical inputs.

**Rule-based evaluation** measures observable properties,
such as formatting and numerical preservation.

**Weighted scoring** assigns different importance to
individual evaluation requirements.

**Held-out evaluation** tests prompt performance on
examples not used during prompt development.

### Evaluation Criteria

All generated posts target X and must satisfy:

| Requirement | Weight |
|---|---:|
| Preserve all source numbers | 40% |
| Maximum 180 characters | 20% |
| Exactly one hashtag | 15% |
| End with a question mark | 15% |
| No emojis | 10% |

The scores measure compliance with these rules,
not complete factual accuracy or writing quality.

The optimized prompt is compared against a baseline;
the notebook does not claim to demonstrate an
automatically discovered or statistically optimal prompt.


In [13]:

import json
import re
import pandas as pd

DEV_CASES = [
    {"id": "D1", "fact": "Customer support response time improved by 18% in April."},
    {"id": "D2", "fact": "The onboarding tutorial was completed by 120 customers in its first week."},
    {"id": "D3", "fact": "Average deployment time decreased from 6 hours to 3 hours."},
    {"id": "D4", "fact": "The company trained 45 employees in data literacy."}
]

TEST_CASES = [
    {"id": "T1", "fact": "Mobile checkout errors decreased by 27% in May."},
    {"id": "T2", "fact": "The customer knowledge base received 350 new articles."},
    {"id": "T3", "fact": "Average ticket resolution time decreased from 10 hours to 7 hours."},
    {"id": "T4", "fact": "The internal mentoring programme enrolled 64 employees."}
]

all_cases = DEV_CASES + TEST_CASES

assert len({case["id"] for case in all_cases}) == 8

display(pd.DataFrame([
    {"split": split, **case}
    for split, cases in [
        ("Development", DEV_CASES),
        ("Test", TEST_CASES)
    ]
    for case in cases
]))


,split,id,fact
0,Development,D1,Customer support response time improved by 18%...
1,Development,D2,The onboarding tutorial was completed by 120 c...
2,Development,D3,Average deployment time decreased from 6 hours...
3,Development,D4,The company trained 45 employees in data liter...
4,Test,T1,Mobile checkout errors decreased by 27% in May.
5,Test,T2,The customer knowledge base received 350 new a...
6,Test,T3,Average ticket resolution time decreased from ...
7,Test,T4,The internal mentoring programme enrolled 64 e...


In [16]:

%pip install -q "openai>=1,<3"

from getpass import getpass
from openai import OpenAI

MODEL = "gemini-3.5-flash-lite"

client = OpenAI(
    api_key=getpass("Gemini API key: "),
    base_url=(
        "https://generativelanguage.googleapis.com/"
        "v1beta/openai/"
    ),
    max_retries=0,
    timeout=120
)

def generate_posts(prompt, cases):
    response = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": prompt},
            {
                "role": "user",
                "content": json.dumps(cases)
            }
        ]
    )

    text = response.choices[0].message.content

    if not text:
        raise ValueError("Empty model response")

    text = text.strip()
    text = re.sub(
        r"^```(?:json)?\s*|\s*```$",
        "",
        text
    ).strip()

    result = json.loads(text)

    expected = {case["id"] for case in cases}
    posts = {}

    if not isinstance(result.get("posts"), list):
        raise ValueError("Missing posts list")

    for item in result["posts"]:
        case_id = item["id"]
        post = item["post"]

        if case_id in posts or case_id not in expected:
            raise ValueError(f"Invalid ID: {case_id}")

        if not isinstance(post, str) or not post.strip():
            raise ValueError(f"Invalid post: {case_id}")

        posts[case_id] = post.strip()

    if set(posts) != expected:
        raise ValueError("Missing post IDs")

    return posts

print("Model:", MODEL)


Gemini API key: ··········
Model: gemini-3.5-flash-lite


In [17]:

BASELINE_PROMPT = """
You are a social media writer.

Create one professional X post for each business update.
Use the supplied facts.

Return JSON only:
{
  "posts": [
    {"id": "D1", "post": "Post text"}
  ]
}

Include every input ID exactly once.
"""

STRUCTURED_PROMPT = """
You are a professional B2B social media editor.

Create one concise X post for each business update.

FACTUAL RULES
- Use only information from the supplied update.
- Preserve every numerical value, unit and comparison.
- Do not introduce numbers or unsupported results.
- Do not invent explanations for the results.

FORMAT RULES
- Maximum 180 characters for the complete post.
- Include exactly one relevant hashtag.
- Place the hashtag before the final question.
- End with a short, relevant question.
- The final character must be ?.
- Do not use emojis.

STYLE
- Clear, natural, professional.
- Avoid exaggeration and generic promotional claims.

Return JSON only:
{
  "posts": [
    {"id": "D1", "post": "Post text"}
  ]
}

Include each supplied ID exactly once.
The IDs in the example illustrate format only.
"""


In [18]:

baseline_dev = generate_posts(
    BASELINE_PROMPT, DEV_CASES
)

structured_dev = generate_posts(
    STRUCTURED_PROMPT, DEV_CASES
)

for case in DEV_CASES:
    case_id = case["id"]

    print(f"\n{case_id} — {case['fact']}")
    print("Baseline:", baseline_dev[case_id])
    print("Structured:", structured_dev[case_id])



D1 — Customer support response time improved by 18% in April.
Baseline: We are dedicated to providing top-notch service! Thanks to our amazing team, our customer support response time improved by 18% in April. 🚀 #CustomerSuccess #Support
Structured: Customer support response time improved by 18% in April. #CustomerSupport How is your team tracking speed?

D2 — The onboarding tutorial was completed by 120 customers in its first week.
Baseline: Off to a fantastic start! Our new onboarding tutorial was successfully completed by 120 customers in its very first week. Thank you for diving right in! 💡 #UserExperience #Onboarding
Structured: The onboarding tutorial was completed by 120 customers in its first week. #Onboarding Are you driving user adoption?

D3 — Average deployment time decreased from 6 hours to 3 hours.
Baseline: Speed and efficiency matter. We're thrilled to announce that our average deployment time has been cut in half, decreasing from 6 hours down to 3 hours! ⚡ #DevOps #En

In [19]:

import unicodedata
from collections import Counter

WEIGHTS = {
    "Numbers": 0.40,
    "Length": 0.20,
    "Hashtag": 0.15,
    "Final question": 0.15,
    "No emojis": 0.10
}

NUMBER_PATTERN = r"\d+(?:[.,]\d+)?%?"


def check_post(case, post):
    source_numbers = Counter(
        re.findall(NUMBER_PATTERN, case["fact"])
    )
    output_numbers = Counter(
        re.findall(NUMBER_PATTERN, post)
    )

    hashtag_count = len(re.findall(
        r"(?<!\w)#[A-Za-z][A-Za-z0-9_]*",
        post
    ))

    has_emoji = any(
        ord(char) >= 0x1F000
        and unicodedata.category(char) == "So"
        for char in post
    )

    return {
        "Numbers": source_numbers == output_numbers,
        "Length": len(post) <= 180,
        "Hashtag": hashtag_count == 1,
        "Final question": post.endswith("?"),
        "No emojis": not has_emoji
    }


def evaluate_versions(cases, versions):
    rows = []

    for version, posts in versions.items():
        for case in cases:
            case_id = case["id"]
            checks = check_post(
                case, posts[case_id]
            )

            score = sum(
                WEIGHTS[criterion] * int(passed)
                for criterion, passed in checks.items()
            )

            rows.append({
                "Version": version,
                "Case": case_id,
                **checks,
                "Score": score
            })

    return pd.DataFrame(rows)


def summarize_scores(df):
    columns = list(WEIGHTS) + ["Score"]

    return (
        df.groupby("Version")[columns]
        .mean()
        .mul(100)
        .round(1)
        .reindex(["Baseline", "Structured"])
    )


assert abs(sum(WEIGHTS.values()) - 1) < 1e-9

print("Evaluation criteria configured.")


Evaluation criteria configured.


In [20]:

dev_results = evaluate_versions(
    DEV_CASES,
    {
        "Baseline": baseline_dev,
        "Structured": structured_dev
    }
)

print("DEVELOPMENT RESULTS")
display(summarize_scores(dev_results))

print("\nFAILED CHECKS")
for _, row in dev_results.iterrows():
    failures = [
        name for name in WEIGHTS
        if not row[name]
    ]

    if failures:
        print(
            row["Version"],
            row["Case"],
            ":", ", ".join(failures)
        )


DEVELOPMENT RESULTS


,Numbers,Length,Hashtag,Final question,No emojis,Score
Version,,,,,,
Baseline,100.0,75.0,0.0,0.0,25.0,57.5
Structured,100.0,100.0,100.0,100.0,100.0,100.0



FAILED CHECKS
Baseline D1 : Hashtag, Final question, No emojis
Baseline D2 : Length, Hashtag, Final question, No emojis
Baseline D3 : Hashtag, Final question
Baseline D4 : Hashtag, Final question, No emojis



## 2. Development Evaluation

The optimized prompt achieved **100% weighted compliance**
on the four development cases, compared with **57.5%**
for the baseline.

The baseline frequently omitted hashtags and closing
questions, and occasionally exceeded the length limit
or included emojis.

The optimized prompt satisfied all five automated
criteria on the development cases.

These findings establish improved compliance with
the specified constraints, but not necessarily
better editorial quality or semantic accuracy.



## 3. Held-Out Evaluation

Both prompts are applied to four additional
business updates not used to design the prompts.

The same scoring rules are used without modification.

This tests whether the optimized prompt's improved
constraint compliance also appears on unseen examples.

The test is exploratory because the dataset is small
and each prompt is evaluated once per case.


In [21]:

baseline_test = generate_posts(
    BASELINE_PROMPT, TEST_CASES
)

structured_test = generate_posts(
    STRUCTURED_PROMPT, TEST_CASES
)

test_results = evaluate_versions(
    TEST_CASES,
    {
        "Baseline": baseline_test,
        "Structured": structured_test
    }
)

print("TEST RESULTS")
display(summarize_scores(test_results))

print("\nDEVELOPMENT VS. TEST")
comparison = pd.concat(
    [
        summarize_scores(dev_results)
        .assign(Split="Development"),
        summarize_scores(test_results)
        .assign(Split="Test")
    ]
)

display(comparison.reset_index())

print("\nTEST OUTPUTS")
for case in TEST_CASES:
    print(f"\n{case['id']}: {case['fact']}")
    print("Baseline:", baseline_test[case["id"]])
    print("Structured:", structured_test[case["id"]])


TEST RESULTS


,Numbers,Length,Hashtag,Final question,No emojis,Score
Version,,,,,,
Baseline,100.0,25.0,0.0,0.0,100.0,55.0
Structured,100.0,100.0,100.0,100.0,100.0,100.0



DEVELOPMENT VS. TEST


,Version,Numbers,Length,Hashtag,Final question,No emojis,Score,Split
0,Baseline,100.0,75.0,0.0,0.0,25.0,57.5,Development
1,Structured,100.0,100.0,100.0,100.0,100.0,100.0,Development
2,Baseline,100.0,25.0,0.0,0.0,100.0,55.0,Test
3,Structured,100.0,100.0,100.0,100.0,100.0,100.0,Test



TEST OUTPUTS

T1: Mobile checkout errors decreased by 27% in May.
Baseline: We are continuously improving your mobile experience! In May, mobile checkout errors decreased by 27%, making transactions smoother and more reliable than ever. #Ecommerce #TechUpdates
Structured: Mobile checkout errors decreased by 27% in May. #Ecommerce Did your mobile conversion rates improve last month?

T2: The customer knowledge base received 350 new articles.
Baseline: Empowering our users with better resources: our customer knowledge base has just been expanded with 350 brand-new articles to help you find answers faster. #CustomerSuccess #Support
Structured: The customer knowledge base received 350 new articles. #CustomerSupport Are you expanding your self-service resources?

T3: Average ticket resolution time decreased from 10 hours to 7 hours.
Baseline: Faster support when you need it most. We've successfully reduced our average ticket resolution time from 10 hours down to 7 hours. #CustomerService #

In [22]:

audit_rows = []

for split, cases, baseline, structured in [
    ("Development", DEV_CASES, baseline_dev, structured_dev),
    ("Test", TEST_CASES, baseline_test, structured_test)
]:
    for case in cases:
        for version, posts in [
            ("Baseline", baseline),
            ("Structured", structured)
        ]:
            audit_rows.append({
                "Split": split,
                "Case": case["id"],
                "Version": version,
                "Source": case["fact"],
                "Generated post": posts[case["id"]]
            })

audit_df = pd.DataFrame(audit_rows)

display(audit_df)

print("MANUAL REVIEW")
print("- Are numbers used in the correct context?")
print("- Are any achievements or causes invented?")
print("- Is the question relevant?")
print("- Does the post sound professional and natural?")


,Split,Case,Version,Source,Generated post
0,Development,D1,Baseline,Customer support response time improved by 18%...,We are dedicated to providing top-notch servic...
1,Development,D1,Structured,Customer support response time improved by 18%...,Customer support response time improved by 18%...
2,Development,D2,Baseline,The onboarding tutorial was completed by 120 c...,Off to a fantastic start! Our new onboarding t...
3,Development,D2,Structured,The onboarding tutorial was completed by 120 c...,The onboarding tutorial was completed by 120 c...
4,Development,D3,Baseline,Average deployment time decreased from 6 hours...,Speed and efficiency matter. We're thrilled to...
5,Development,D3,Structured,Average deployment time decreased from 6 hours...,Average deployment time decreased from 6 hours...
6,Development,D4,Baseline,The company trained 45 employees in data liter...,Empowering our team from the ground up: we rec...
7,Development,D4,Structured,The company trained 45 employees in data liter...,The company trained 45 employees in data liter...
8,Test,T1,Baseline,Mobile checkout errors decreased by 27% in May.,We are continuously improving your mobile expe...
9,Test,T1,Structured,Mobile checkout errors decreased by 27% in May.,Mobile checkout errors decreased by 27% in May...


MANUAL REVIEW
- Are numbers used in the correct context?
- Are any achievements or causes invented?
- Is the question relevant?
- Does the post sound professional and natural?



## Findings and Limitations

### What This Notebook Demonstrates

- Comparing baseline and optimized prompts.
- Applying explicit formatting and factual constraints.
- Evaluating real Gemini-generated outputs.
- Measuring performance with deterministic Python checks.
- Calculating weighted compliance scores.
- Comparing development and held-out test results.
- Identifying limitations of automated evaluation.

### Prompt Engineering Techniques

**Structured Instructions**

The optimized prompt specifies the role, task,
factual requirements, output format and constraints.

**Grounding**

Generated posts are restricted to the supplied
business updates to reduce unsupported claims.

**A/B Testing**

Both prompt versions are evaluated using identical
inputs, model settings and scoring criteria.

**Weighted Evaluation**

Requirements receive predefined weights based
on their importance to the task.

**Held-Out Testing**

Separate test examples assess whether improved
compliance extends beyond development cases.

### Relationship to the Udemy Examples

This notebook adapts prompt refinement and
evaluation techniques from:

- `social_media_posts.ipynb`
- `evaluation_metrics.ipynb`

The focus is on reproducible, rule-based evaluation
rather than subjective impressions of output quality.

LLM-as-a-judge and automated DSPy optimization are
not implemented in this notebook.

### Observed Results

| Metric | Dev Baseline | Dev Optimized | Test Baseline | Test Optimized |
|---|---:|---:|---:|---:|
| Weighted compliance | 57.5% | 100% | 55% | 100% |
| Numerical preservation | 100% | 100% | 100% | 100% |
| Length compliance | 75% | 100% | 25% | 100% |
| Hashtag compliance | 0% | 100% | 0% | 100% |
| Final-question compliance | 0% | 100% | 0% | 100% |
| No-emoji compliance | 25% | 100% | 100% | 100% |

The optimized prompt satisfied all automated
criteria across both development and test cases.

The baseline preserved numerical values but
frequently failed formatting requirements.

**Manual Review**

The optimized posts generally reflected the
source updates accurately.

However, some closing questions introduced
topics not established by the source. For example,
a question about conversion rates appeared in
response to an update about checkout errors.

This did not necessarily create a false claim,
but demonstrated a limitation of the automated
compliance checks.

### Limitations

- Only eight fictional examples were evaluated.
- Each prompt was executed once per case.
- Evaluation weights were manually selected.
- Numerical matching does not guarantee
  complete factual accuracy.
- Rule-based checks cannot fully assess
  relevance, tone or unsupported implications.
- The prompts differ in multiple instructions,
  preventing attribution to one specific change.
- The test set is small and not statistically
  representative.
- No automated semantic or human-rated
  quality benchmark was implemented.

### Conclusion

The optimized prompt increased weighted compliance
from **57.5% to 100%** on development cases and
from **55% to 100%** on held-out test cases.

The experiment demonstrates that explicit
instructions can improve measurable constraint
adherence.

However, successful automated evaluation does not
guarantee complete factual accuracy or editorial
quality. Reliable prompt evaluation requires both
measurable checks and critical interpretation
of the generated outputs.
